In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
!pip install -U "bitsandbytes>=0.46.1" -q
!pip install -U torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 74.1 MB/s eta 0:00:00


In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/model.safetensors.index.json
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/config.json
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/model-00001-of-00002.safetensors
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/model-00002-of-00002.safetensors
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/README.md
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/tokenizer.json
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/tokenizer_config.json
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/special_tokens_map.json
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/.gitattributes
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/tokenizer.model
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/generation_config.json
/kaggle/input/models/google/gemma/transformers/2b-it/

In [4]:
base = "/kaggle/input/competitions/agriculture-climate-slm-challenge/"
documents = pd.read_csv(base + "documents.csv")
train_qa = pd.read_csv(base + "train_qa.csv")
test_questions = pd.read_csv(base + "test_questions.csv")

for name, df in [("documents", documents), ("train_qa", train_qa), ("test_questions", test_questions)]:
    print(f"--- {name} ---")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print(df.dtypes)
    print(df.isna().sum())
    print()

--- documents ---
shape: (24, 9)
columns: ['document_id', 'title', 'topic', 'crop', 'agro_zone', 'text', 'origin', 'source_url', 'license']
document_id     object
title           object
topic           object
crop            object
agro_zone       object
text            object
origin          object
source_url     float64
license         object
dtype: object
document_id     0
title           0
topic           0
crop            0
agro_zone       0
text            0
origin          0
source_url     24
license         0
dtype: int64

--- train_qa ---
shape: (45, 7)
columns: ['question', 'topic', 'crop', 'agro_zone', 'document_id', 'reference_answer', 'QuestionId']
question            object
topic               object
crop                object
agro_zone           object
document_id         object
reference_answer    object
QuestionId           int64
dtype: object
question            0
topic               0
crop                0
agro_zone           0
document_id         0
reference_answer 

In [5]:
# Does (topic, crop, agro_zone) uniquely map to a document_id?
doc_key_counts = documents.groupby(["topic", "crop", "agro_zone"])["document_id"].nunique()
print("Max documents sharing the same (topic, crop, agro_zone):", doc_key_counts.max())
print(doc_key_counts.sort_values(ascending=False).head(10))
print()

# How many training questions per document?
print("Questions per document_id (train_qa):")
print(train_qa["document_id"].value_counts())
print()

# Does train_qa's (topic, crop, agro_zone) match its linked document's own metadata?
merged_check = train_qa.merge(documents, on="document_id", suffixes=("_qa", "_doc"))
mismatch = merged_check[
    (merged_check["topic_qa"] != merged_check["topic_doc"]) |
    (merged_check["crop_qa"] != merged_check["crop_doc"]) |
    (merged_check["agro_zone_qa"] != merged_check["agro_zone_doc"])
]
print("Rows where QA metadata != linked document metadata:", len(mismatch))
print()

# Category coverage: are test categories a subset of train/document categories?
for col in ["topic", "crop", "agro_zone"]:
    doc_vals = set(documents[col].unique())
    train_vals = set(train_qa[col].unique())
    test_vals = set(test_questions[col].unique())
    print(f"{col}: doc={len(doc_vals)} train={len(train_vals)} test={len(test_vals)}")
    print(f"  test values not in documents: {test_vals - doc_vals}")
    print(f"  test values not in train_qa: {test_vals - train_vals}")

Max documents sharing the same (topic, crop, agro_zone): 2
topic               crop       agro_zone
livestock           livestock  sub_humid    2
climate_adaptation  livestock  semi_arid    1
                    general    semi_arid    1
crop_diseases       beans      highland     1
                    cassava    sub_humid    1
                    maize      sub_humid    1
climate_adaptation  maize      semi_arid    1
fertiliser          beans      highland     1
                    general    sub_humid    1
                    maize      sub_humid    1
Name: document_id, dtype: int64

Questions per document_id (train_qa):
document_id
doc_soi_002    3
doc_fer_001    3
doc_pes_001    3
doc_liv_001    3
doc_dis_003    3
doc_cli_003    2
doc_pos_001    2
doc_soi_001    2
doc_pos_003    2
doc_dis_002    2
doc_liv_003    2
doc_dis_001    2
doc_wat_002    2
doc_fer_003    2
doc_wat_003    2
doc_cli_002    2
doc_pes_003    2
doc_soi_003    1
doc_wat_001    1
doc_pos_002    1
doc_cli_001    1


In [6]:
def char_stats(series, label):
    lens = series.str.len()
    print(f"{label}: n={len(lens)}  min={lens.min()}  mean={lens.mean():.0f}  median={lens.median():.0f}  max={lens.max()}")

char_stats(documents["text"], "document text (chars)")
char_stats(train_qa["question"], "train question (chars)")
char_stats(train_qa["reference_answer"], "train reference_answer (chars)")
char_stats(test_questions["question"], "test question (chars)")
print()

# word counts for reference answers specifically (closer to what generation length should target)
word_lens = train_qa["reference_answer"].str.split().str.len()
print("reference_answer word count: min={} mean={:.1f} median={} max={}".format(
    word_lens.min(), word_lens.mean(), word_lens.median(), word_lens.max()))
print()

# Look at a handful of full examples to judge extractive vs abstractive style
for i in [0, 1, 2]:
    row = train_qa.iloc[i]
    doc_text = documents.loc[documents["document_id"] == row["document_id"], "text"].values[0]
    print(f"Q: {row['question']}")
    print(f"REFERENCE ANSWER: {row['reference_answer']}")
    print(f"Is reference_answer a verbatim substring of doc text? {row['reference_answer'] in doc_text}")
    print(f"DOC TEXT (first 400 chars): {doc_text[:400]}")
    print("-" * 80)

document text (chars): n=24  min=204  mean=254  median=256  max=295
train question (chars): n=45  min=27  mean=42  median=42  max=76
train reference_answer (chars): n=45  min=51  mean=68  median=68  max=93
test question (chars): n=12  min=29  mean=43  median=40  max=59

reference_answer word count: min=7 mean=10.3 median=10.0 max=14

Q: Weevils in stored maize without chemicals?
REFERENCE ANSWER: Dry to twelve to thirteen percent and seal in hermetic bags.
Is reference_answer a verbatim substring of doc text? False
DOC TEXT (first 400 chars): Hermetic bags or silos exclude oxygen and stop maize weevils without chemical dust. Dry grain to twelve to thirteen percent moisture before sealing. Inspect bags monthly for punctures and reseal promptly.
--------------------------------------------------------------------------------
Q: Insurance paid but my field still failed — why?
REFERENCE ANSWER: Basis risk means index payouts may not match individual field losses.
Is reference_answer a verb

In [7]:
# 1. Look at the ambiguous key: livestock/livestock/sub_humid maps to 2 documents
ambig = documents[(documents["topic"] == "livestock") & (documents["crop"] == "livestock") & (documents["agro_zone"] == "sub_humid")]
print(ambig[["document_id", "title", "text"]])
print()

# 2. Does this ambiguous combo appear in test_questions?
test_ambig = test_questions[(test_questions["topic"] == "livestock") & (test_questions["crop"] == "livestock") & (test_questions["agro_zone"] == "sub_humid")]
print("Ambiguous combo rows in test:", len(test_ambig))
print(test_ambig)
print()

# 3. Full spread of test_questions metadata, to plan doc-matching for all 12
print(test_questions[["QuestionId", "question", "topic", "crop", "agro_zone"]].to_string())

    document_id                                  title  \
18  doc_liv_001  Newcastle disease in village chickens   
20  doc_liv_003       Tsetse-free corralling practices   

                                                 text  
18  Newcastle disease causes respiratory distress,...  
20  Night corralling in smoke-treated pens reduces...  

Ambiguous combo rows in test: 0
Empty DataFrame
Columns: [QuestionId, question, topic, crop, agro_zone]
Index: []

    QuestionId                                                     question               topic        crop  agro_zone
0         1001  How should I apply nitrogen to leaching-prone maize fields?       crop_diseases       maize  sub_humid
1         1002                        Grass gone in August — feed strategy?           livestock   livestock  semi_arid
2         1003             Red spots under my bean leaves during the rains.       crop_diseases       beans   highland
3         1004                     Fresh cow dung on vegetable be

In [8]:
def match_document(row, documents):
    matches = documents[
        (documents["topic"] == row["topic"]) &
        (documents["crop"] == row["crop"]) &
        (documents["agro_zone"] == row["agro_zone"])
    ]
    return matches

results = []
for _, row in test_questions.iterrows():
    matches = match_document(row, documents)
    results.append({
        "QuestionId": row["QuestionId"],
        "question": row["question"],
        "n_matches": len(matches),
        "matched_doc_ids": list(matches["document_id"])
    })

match_df = pd.DataFrame(results)
print(match_df.to_string())
print()
print("Any test question with 0 or >1 matches?")
print(match_df[match_df["n_matches"] != 1])

    QuestionId                                                     question  n_matches matched_doc_ids
0         1001  How should I apply nitrogen to leaching-prone maize fields?          1   [doc_dis_001]
1         1002                        Grass gone in August — feed strategy?          1   [doc_liv_002]
2         1003             Red spots under my bean leaves during the rains.          1   [doc_dis_002]
3         1004                     Fresh cow dung on vegetable beds — safe?          1   [doc_fer_002]
4         1005                                How much compost per hectare?          1   [doc_fer_002]
5         1006                         How can I lower stem borer pressure?          1   [doc_pes_002]
6         1007                    Legumes not nodulating on my acidic plot.          1   [doc_soi_001]
7         1008                Sorghum seedlings dying at the growing point.          1   [doc_pes_002]
8         1009                            Compost pile smells rotten — fi

In [9]:
# Validate: does metadata-matching recover the correct document_id for train_qa too?
def match_document_id(row, documents):
    matches = documents[
        (documents["topic"] == row["topic"]) &
        (documents["crop"] == row["crop"]) &
        (documents["agro_zone"] == row["agro_zone"])
    ]
    if len(matches) == 1:
        return matches["document_id"].values[0]
    elif len(matches) > 1:
        return "AMBIGUOUS:" + ",".join(matches["document_id"])
    else:
        return "NO_MATCH"

train_qa["matched_doc_id"] = train_qa.apply(lambda r: match_document_id(r, documents), axis=1)
train_qa["match_correct"] = train_qa["matched_doc_id"] == train_qa["document_id"]
print("Metadata-match accuracy on train_qa:", train_qa["match_correct"].mean())
print(train_qa[~train_qa["match_correct"]][["QuestionId", "document_id", "matched_doc_id", "topic", "crop", "agro_zone"]])
print()

# Sample 10 more QA pairs across different topics to study answer style/phrasing patterns
sample = train_qa.sample(10, random_state=42)
for _, row in sample.iterrows():
    doc_text = documents.loc[documents["document_id"] == row["document_id"], "text"].values[0]
    print(f"[{row['topic']}] Q: {row['question']}")
    print(f"A: {row['reference_answer']}")
    print(f"DOC: {doc_text}")
    print("-" * 80)

Metadata-match accuracy on train_qa: 0.8888888888888888
    QuestionId  document_id                     matched_doc_id      topic  \
11          12  doc_liv_003  AMBIGUOUS:doc_liv_001,doc_liv_003  livestock   
14          15  doc_liv_003  AMBIGUOUS:doc_liv_001,doc_liv_003  livestock   
17          18  doc_liv_001  AMBIGUOUS:doc_liv_001,doc_liv_003  livestock   
34          35  doc_liv_001  AMBIGUOUS:doc_liv_001,doc_liv_003  livestock   
44          45  doc_liv_001  AMBIGUOUS:doc_liv_001,doc_liv_003  livestock   

         crop  agro_zone  
11  livestock  sub_humid  
14  livestock  sub_humid  
17  livestock  sub_humid  
34  livestock  sub_humid  
44  livestock  sub_humid  

[pests] Q: Sticky soot on seedlings in the nursery.
A: Aphid honeydew leading to sooty mould; wash aphids and protect natural enemies.
DOC: Aphids cluster on tender shoots causing leaf curl and honeydew that leads to sooty mould. Monitor nursery beds daily, use reflective mulch, and wash colonies off with soapy water

In [10]:
def build_prompt(question, doc_text):
    return (
        "You are an agricultural extension assistant. Using ONLY the factsheet below, "
        "answer the farmer's question in ONE short sentence (about 8-14 words). "
        "Be direct and specific, matching the factsheet's terminology.\n\n"
        f"Factsheet: {doc_text}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

# Build training pairs: prompt -> target answer
train_qa = train_qa.merge(documents[["document_id", "text"]], on="document_id", how="left")
train_qa["prompt"] = train_qa.apply(lambda r: build_prompt(r["question"], r["text"]), axis=1)
train_qa["target"] = train_qa["reference_answer"]

# Sanity check on one example
print(train_qa.iloc[0]["prompt"])
print("\n>>> TARGET:", train_qa.iloc[0]["target"])
print("\nPrompt char length stats:")
print(train_qa["prompt"].str.len().describe())

You are an agricultural extension assistant. Using ONLY the factsheet below, answer the farmer's question in ONE short sentence (about 8-14 words). Be direct and specific, matching the factsheet's terminology.

Factsheet: Hermetic bags or silos exclude oxygen and stop maize weevils without chemical dust. Dry grain to twelve to thirteen percent moisture before sealing. Inspect bags monthly for punctures and reseal promptly.

Question: Weevils in stored maize without chemicals?
Answer:

>>> TARGET: Dry to twelve to thirteen percent and seal in hermetic bags.

Prompt char length stats:
count     45.000000
mean     541.177778
std       30.476734
min      476.000000
25%      528.000000
50%      536.000000
75%      559.000000
max      606.000000
Name: prompt, dtype: float64


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model_path = "/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2"

tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    device_map={"": 0},
)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

trainable params: 6,389,760 || all params: 2,620,731,648 || trainable%: 0.2438


In [12]:
from datasets import Dataset

MAX_LEN = 256  

def tokenize_example(example):
    prompt = example["prompt"]
    target = " " + example["target"] + tokenizer.eos_token

    prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
    target_ids = tokenizer(target, add_special_tokens=False)["input_ids"]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids  # mask prompt, learn only target

    input_ids = input_ids[:MAX_LEN]
    labels = labels[:MAX_LEN]

    return {"input_ids": input_ids, "labels": labels}

train_dataset = Dataset.from_pandas(train_qa[["prompt", "target"]])
train_dataset = train_dataset.map(tokenize_example, remove_columns=["prompt", "target"])

# Sanity check: decode labels back to confirm masking is correct
example = train_dataset[0]
visible_labels = [t for t in example["labels"] if t != -100]
print("Decoded target-only labels:", tokenizer.decode(visible_labels))
print("Total input length:", len(example["input_ids"]))

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

Decoded target-only labels:  Dry to twelve to thirteen percent and seal in hermetic bags.<eos>
Total input length: 118


In [13]:
def data_collator(features):
    max_len = max(len(f["input_ids"]) for f in features)
    pad_id = tokenizer.pad_token_id

    input_ids, labels, attention_mask = [], [], []
    for f in features:
        pad_len = max_len - len(f["input_ids"])
        input_ids.append(f["input_ids"] + [pad_id] * pad_len)
        labels.append(f["labels"] + [-100] * pad_len)
        attention_mask.append([1] * len(f["input_ids"]) + [0] * pad_len)

    return {
        "input_ids": torch.tensor(input_ids),
        "labels": torch.tensor(labels),
        "attention_mask": torch.tensor(attention_mask),
    }

In [14]:
from transformers import Trainer, TrainingArguments

model.config.use_cache = False
model.gradient_checkpointing_enable()

training_args = TrainingArguments(
    output_dir="/kaggle/working/lora_out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=8,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss
5,1.935888
10,0.978204
15,0.620113
20,0.328992
25,0.245776
30,0.100373
35,0.044490
40,0.025695
45,0.013354


TrainOutput(global_step=48, training_loss=0.4476048063176374, metrics={'train_runtime': 411.135, 'train_samples_per_second': 0.876, 'train_steps_per_second': 0.117, 'total_flos': 558873290207232.0, 'train_loss': 0.4476048063176374, 'epoch': 8.0})

In [15]:
def generate_answer(question, doc_text, max_new_tokens=30):
    prompt = build_prompt(question, doc_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

model.config.use_cache = True  # re-enable for generation (was off for training)
model.eval()

# Check a few TRAIN examples: does it reproduce reference_answer closely? (expected, given near-zero loss)
print("=== TRAIN sample checks ===")
for i in [0, 5, 20]:
    row = train_qa.iloc[i]
    pred = generate_answer(row["question"], row["text"])
    print(f"Q: {row['question']}")
    print(f"REF : {row['reference_answer']}")
    print(f"PRED: {pred}")
    print("-" * 80)

# Now the real test: TEST questions, using metadata-matched documents
test_questions_merged = test_questions.merge(
    documents[["topic", "crop", "agro_zone", "document_id", "text"]],
    on=["topic", "crop", "agro_zone"], how="left"
)

print("=== TEST predictions ===")
for _, row in test_questions_merged.iterrows():
    pred = generate_answer(row["question"], row["text"])
    print(f"QuestionId {row['QuestionId']}: {row['question']}")
    print(f"PRED: {pred}")
    print("-" * 80)

=== TRAIN sample checks ===
Q: Weevils in stored maize without chemicals?
REF : Dry to twelve to thirteen percent and seal in hermetic bags.
PRED: Dry to twelve to thirteen percent and seal in hermetic bags.
--------------------------------------------------------------------------------
Q: Can I apply urea on dry soil?
REF : Only if rain or irrigation is imminent; otherwise uptake is poor.
PRED: Only if rain or irrigation is imminent; otherwise uptake is poor.
--------------------------------------------------------------------------------
Q: Why does low pH hurt legumes?
REF : Acidity binds phosphorus and limits nodulation below about pH 5.5.
PRED: Acidity binds phosphorus and limits nodulation below about pH 5.5.
--------------------------------------------------------------------------------
=== TEST predictions ===
QuestionId 1001: How should I apply nitrogen to leaching-prone maize fields?
PRED: Split uptake at planting and knee-high with balanced N;
-----------------------------

In [16]:
predictions = []
for _, row in test_questions_merged.iterrows():
    pred = generate_answer(row["question"], row["text"])
    predictions.append({"QuestionId": row["QuestionId"], "Answer": pred})

submission = pd.DataFrame(predictions)

# Preserve original test_questions order exactly
submission = submission.set_index("QuestionId").loc[test_questions["QuestionId"]].reset_index()

# Validation checks per the competition rules
assert list(submission.columns) == ["QuestionId", "Answer"], "Column mismatch"
assert len(submission) == 12, f"Expected 12 rows, got {len(submission)}"
assert list(submission["QuestionId"]) == list(test_questions["QuestionId"]), "Order mismatch"
assert submission["Answer"].str.strip().str.len().gt(0).all(), "Empty answer found"

print("All checks passed.")
print(submission)

submission.to_csv("/kaggle/working/submission1.csv", index=False)

All checks passed.
    QuestionId                                             Answer
0         1001  Split uptake at planting and knee-high with ba...
1         1002  Harvest forage at boot stage, sun-dry on racks...
2         1003  Bean rust — remove infected debris, avoid dusk...
3         1004  Avoid fresh manure on leafy vegetables; use co...
4         1005  Five to ten tonnes before planting, avoiding f...
5         1006  Destroy residues after harvest and plant early...
6         1007  Acidity above pH 5.5 binds phosphorus and limi...
7         1008  Stem borer tunneling; destroy residues and pla...
8         1009  Ensure it's well-mixed with balanced green and...
9         1010  Align with rainfall onset using calendars or f...
10        1011  Damped groundnuts left on bare soil raise afla...
11        1012  Channel roof runoff into screened tanks with f...
